# 04 — Metrics & Reconciliation

M1-M5 computed by `pipeline/metrics.py` against the real 12-month backfill, and the reconciliation exercise that resolves "which clock?" empirically. Full writeup: `docs/judgement_call.md`. Run `python -m pipeline.metrics` before this notebook.


In [1]:
import json, pandas as pd
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
metrics = json.loads((REPO_ROOT / "outputs" / "metrics.json").read_text())
print("target:", metrics["target_pct"], "in", metrics["target_minutes"], "minutes")


target: 0.9 in 10 minutes


## M1 — reconciliation: which definition best reproduces the official number?


In [2]:
recon = pd.DataFrame(metrics["m1_reconciliation"]).T.sort_values("mean_absolute_gap")
recon


,months_compared,mean_absolute_gap
dispatch_original_medic,12.0,0.0351
dispatch_ctg_any_responder,12.0,0.0358
dispatch_final_ambulance,12.0,0.0501
received_original_medic,12.0,0.2161
received_final_ambulance,12.0,0.2208


`dispatch_original_medic` (clock=dispatch, priority=original∈{3,E}, units=MEDIC only) wins by a wide margin over every received-clock alternative (3.5 points vs. 20+ points mean absolute gap) — but only narrowly over `dispatch_ctg_any_responder` (3.6 points), which scopes to *any* first responder, not just ambulances. The clock choice dominates; the unit-scope choice is close enough that it's not fully resolved by this data alone. See `docs/judgement_call.md`.


## M1 — best-fitting definition vs. official, month by month


In [3]:
best_id = metrics["m1_best_fitting_definition"]
df = pd.DataFrame(metrics["m1_kpi_monthly"])
best = df[df.definition_id == best_id][["month", "pct"]].set_index("month")
official = pd.Series(metrics["official_scorecard_by_month"], name="official")
comparison = best.join(official).dropna()
comparison["gap_points"] = ((comparison["official"] - comparison["pct"]) * 100).round(1)
comparison


,pct,official,gap_points
month,,,
2025-07,0.8469,0.880,3.3
2025-08,0.8349,0.880,4.5
2025-09,0.8321,0.882,5.0
2025-10,0.8256,0.860,3.4
2025-11,0.8348,0.880,4.5
2025-12,0.8173,0.850,3.3
2026-01,0.8264,0.863,3.7
2026-02,0.8379,0.873,3.5
2026-03,0.8535,0.875,2.1


## M5 — the caller-experience gap (dispatch clock vs. received clock)


In [4]:
pd.DataFrame(metrics["m5_definition_gap"])


,month,m1_received_clock_pct,m1_dispatch_clock_pct,clock_gap_points,official_pct,official_vs_dispatch_gap_points
0,2025-07,0.6703,0.8437,17.3,0.880,3.6
1,2025-08,0.6495,0.8314,18.2,0.880,4.9
2,2025-09,0.6544,0.8222,16.8,0.882,6.0
3,2025-10,0.6503,0.8101,16.0,0.860,5.0
4,2025-11,0.6597,0.8264,16.7,0.880,5.4
5,2025-12,0.6249,0.7983,17.3,0.850,5.2
6,2026-01,0.6283,0.8078,17.9,0.863,5.5
7,2026-02,0.6370,0.8202,18.3,0.873,5.3
8,2026-03,0.6548,0.8230,16.8,0.875,5.2
9,2026-04,0.6705,0.8332,16.3,0.885,5.2


The gap between the two clock choices is consistently **~15 percentage points**, every month — this is not a one-off finding from a single cherry-picked month.


## M2 — call-processing time (received → dispatch)


In [5]:
pd.DataFrame(metrics["m2_call_processing_p50_p90"])


,month,p50_minutes,p90_minutes,n
0,2025-07,2.3,4.9,14911
1,2025-08,2.2,4.8,15363
2,2025-09,2.2,4.7,15268
3,2025-10,2.2,4.7,15659
4,2025-11,2.2,4.7,14544
5,2025-12,2.3,4.9,16340
6,2026-01,2.2,4.8,15723
7,2026-02,2.3,4.9,14354
8,2026-03,2.3,5.0,15633
9,2026-04,2.2,4.8,14666


## M3 — ambulance travel time (en route → on scene, MEDIC/PRIVATE only)


In [6]:
pd.DataFrame(metrics["m3_travel_time_p50_p90"])


,month,p50_minutes,p90_minutes,n
0,2025-07,7.2,17.0,9025
1,2025-08,7.3,17.5,9359
2,2025-09,7.5,18.0,9075
3,2025-10,7.6,18.1,9543
4,2025-11,7.4,17.5,8662
5,2025-12,7.7,18.1,9576
6,2026-01,7.5,17.6,9445
7,2026-02,7.6,17.8,8818
8,2026-03,7.3,17.1,9576
9,2026-04,7.2,16.8,8971


## M4 — hospital turnaround and ambulance-hours lost


In [7]:
m4 = pd.DataFrame(metrics["m4_hospital_turnaround"])
m4


,month,n_transports,n_over_standard,pct_over_standard,p50_minutes,p90_minutes,ambulance_hours_lost
0,2025-07,6913,5702,0.8248,41.0,64.1,1706.8
1,2025-08,7110,5850,0.8228,40.9,64.3,1737.5
2,2025-09,6983,5686,0.8143,40.2,63.4,1669.0
3,2025-10,7241,5983,0.8263,41.4,65.2,1832.7
4,2025-11,6585,5380,0.8170,41.3,63.6,1617.7
5,2025-12,7425,6258,0.8428,42.6,68.0,2062.9
6,2026-01,7429,6270,0.8440,43.5,67.9,2098.4
7,2026-02,6945,5927,0.8534,43.7,68.3,2016.2
8,2026-03,7350,6192,0.8424,42.3,66.9,1959.5
9,2026-04,7048,5912,0.8388,41.9,64.3,1762.5


In [8]:
total_hours = m4["ambulance_hours_lost"].sum()
avg_pct_over = m4["pct_over_standard"].mean()
print(f"Total ambulance-hours lost beyond the {metrics['m4_standard']['turnaround_minutes']}-min "
      f"turnaround standard, 12 months: {total_hours:,.0f}")
print(f"Average share of transports exceeding the standard: {avg_pct_over:.1%}")
print(f"That's roughly {total_hours/12/24:.1f} twelve-hour ambulance shifts per day, system-wide.")


Total ambulance-hours lost beyond the 30-min turnaround standard, 12 months: 21,896
Average share of transports exceeding the standard: 83.1%
That's roughly 76.0 twelve-hour ambulance shifts per day, system-wide.


**~82% of transports exceed the 30-minute turnaround standard**, and the system loses roughly 21,900 ambulance-hours over 12 months beyond that standard — equivalent to several dozen twelve-hour ambulance shifts *per day* that could otherwise be answering 911 calls. This compares against the 30-minute **turnaround** standard (`hospital_dttm → available_dttm`), not the 20-minute **offload** standard — the latter requires a hospital-EHR timestamp this dataset doesn't have (`docs/decision_log.md`, 2026-09-23). The true offload-standard gap is almost certainly larger, since offload is a strict subset of turnaround time, but it is not computed here because this dataset cannot measure it - stated as a limitation, not filled in with a guess.


## Summary

- **M1 (KPI):** best-reproducing definition lands within ~3.5 points of official, every month, using a dispatch clock — resolving the "which clock" judgement call empirically (`docs/judgement_call.md`).
- **M2:** call-processing (received→dispatch) is fast: p50 ~2.3 min, p90 ~4.9 min. Not the bottleneck.
- **M3:** ambulance travel time (en route→on scene) is p50 ~7.2 min, p90 ~17 min.
- **M4:** hospital turnaround is the dominant bottleneck — p50 ~41 min against a 30-min standard, ~82% non-compliant, ~21,900 ambulance-hours lost over 12 months.
- **M5:** the caller-experienced KPI (911-call clock) trails the published dispatch-clock KPI by ~15 points, consistently.
